# Ch01-03: 이상치 탐지 - 통계적 방법 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

5차원 Mahalanobis + QQ + ROC

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis
from sklearn.metrics import roc_curve, auc

np.random.seed(42); p=5
cov_n = np.eye(p);
for i in range(p):
    for j in range(p):
        if i!=j: cov_n[i,j]=0.5**abs(i-j)
X_n = np.random.multivariate_normal(np.zeros(p), cov_n, 500)
X_o = np.random.multivariate_normal(np.ones(p)*4, np.eye(p)*0.2, 20)
X = np.vstack([X_n, X_o]); y = np.array([0]*500+[1]*20)

mu = X.mean(0); Si = np.linalg.inv(np.cov(X.T))
D2 = np.array([mahalanobis(x,mu,Si)**2 for x in X])

fig, axes = plt.subplots(1,2,figsize=(14,5))
sd2 = np.sort(D2); theo = stats.chi2.ppf(np.arange(1,len(sd2)+1)/(len(sd2)+1), p)
axes[0].scatter(theo, sd2, s=8, alpha=0.5); axes[0].plot([0,max(theo)],[0,max(theo)],'r--')
axes[0].set_title('χ²(5) QQ-Plot')
fpr,tpr,_ = roc_curve(y, D2)
axes[1].plot(fpr,tpr,'b-',lw=2,label=f'AUC={auc(fpr,tpr):.3f}')
axes[1].plot([0,1],[0,1],'r--'); axes[1].set_title('ROC'); axes[1].legend()
plt.tight_layout(); plt.show()

thr = stats.chi2.ppf(0.975, p); det = D2>thr
tp=((det)&(y==1)).sum(); fp=((det)&(y==0)).sum()
print(f"Precision={tp/(tp+fp):.3f}, Recall={tp/(tp+(~det&(y==1)).sum()):.3f}")


**결과 해석**: QQ-plot 상위 꼬리 이탈 점이 이상치 후보. AUC가 높을수록 Mahalanobis의 분리 능력이 우수함을 나타낸다.

---
## 문제 2 풀이

Generalized ESD 직접 구현

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats

def generalized_esd(data, max_outliers=15, alpha=0.05):
    data = np.array(data, dtype=float); n = len(data)
    remaining = data.copy(); rem_idx = np.arange(n)
    out_idx, Rs, lams = [], [], []
    for i in range(1, max_outliers+1):
        ni = len(remaining)
        if ni < 3: break
        m, s = np.mean(remaining), np.std(remaining, ddof=1)
        if s == 0: break
        devs = np.abs(remaining - m); mi = np.argmax(devs)
        Ri = devs[mi] / s
        p = 1 - alpha / (2*ni)
        tp = stats.t.ppf(p, ni-2)
        lam_i = (ni-1)*tp / np.sqrt((ni-2+tp**2)*ni)
        Rs.append(Ri); lams.append(lam_i); out_idx.append(rem_idx[mi])
        remaining = np.delete(remaining, mi); rem_idx = np.delete(rem_idx, mi)
    n_out = 0
    for i in range(len(Rs)):
        if Rs[i] > lams[i]: n_out = i+1
    return {'n_outliers': n_out, 'indices': out_idx[:n_out], 'R': Rs, 'lambda': lams}

np.random.seed(42)
data = np.concatenate([np.random.normal(50,5,200), np.array([90,95,85,10,5,88])])
r = generalized_esd(data, 15)
print(f"탐지: {r['n_outliers']}개, 값: {data[r['indices']]}")

fig, ax = plt.subplots(figsize=(10,4))
ax.bar(range(1,len(r['R'])+1), r['R'], alpha=0.7, label='R_i')
ax.plot(range(1,len(r['lambda'])+1), r['lambda'], 'r-o', lw=2, label='λ_i')
ax.axvline(r['n_outliers']+0.5, color='green', ls='--'); ax.legend()
ax.set_title('Generalized ESD'); plt.tight_layout(); plt.show()


**결과 해석**: ESD는 순차적으로 극단값을 제거하며 검정하여 마스킹 효과를 부분적으로 완화한다.

---
## 문제 3 풀이

이상치 비율별 Classical vs MCD 성능 비교

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
from scipy.spatial.distance import mahalanobis
from scipy import stats
from sklearn.metrics import f1_score

np.random.seed(42)
def sim(rate, n=500, p=3, trials=20):
    f1c, f1m = [], []
    for t in range(trials):
        rng = np.random.RandomState(t)
        no = int(n*rate); nn = n-no
        cv = np.eye(p); cv[0,1]=cv[1,0]=0.5
        Xn = rng.multivariate_normal(np.zeros(p), cv, nn)
        Xo = rng.multivariate_normal(np.ones(p)*5, np.eye(p)*0.3, no)
        X = np.vstack([Xn,Xo]); y = np.array([0]*nn+[1]*no)
        Si = np.linalg.pinv(np.cov(X.T)); mu = X.mean(0)
        D2c = np.array([mahalanobis(x,mu,Si)**2 for x in X])
        pc = (D2c>stats.chi2.ppf(0.975,p)).astype(int)
        try:
            mcd = MinCovDet(random_state=t).fit(X)
            D2m = mcd.mahalanobis(X)
        except: D2m = np.zeros(n)
        pm = (D2m>stats.chi2.ppf(0.975,p)).astype(int)
        f1c.append(f1_score(y,pc,zero_division=0)); f1m.append(f1_score(y,pm,zero_division=0))
    return np.mean(f1c), np.mean(f1m)

rates = [0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40]
fc, fm = zip(*[sim(r) for r in rates])
fig, ax = plt.subplots(figsize=(10,5))
ax.plot([r*100 for r in rates], fc, 'bo-', lw=2, label='Classical')
ax.plot([r*100 for r in rates], fm, 'rs-', lw=2, label='MCD')
ax.set_xlabel('이상치 비율 (%)'); ax.set_ylabel('F1'); ax.legend()
ax.set_title('이상치 비율별 F1 비교'); plt.tight_layout(); plt.show()


**결과 해석**: 이상치 비율 증가 시 Classical Mahalanobis는 마스킹으로 F1 급락. MCD는 붕괴점(~50%)까지 안정적 성능 유지.

---
## 문제 4 풀이

마스킹 효과 시각화

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.spatial.distance import mahalanobis
from scipy import stats
from sklearn.covariance import MinCovDet

np.random.seed(42)
X_n = np.random.multivariate_normal([0,0], [[1,0.5],[0.5,1]], 100)
X_o = np.random.multivariate_normal([6,6], [[0.2,0],[0,0.2]], 20)
X = np.vstack([X_n, X_o]); y = np.array([0]*100+[1]*20)

mu = X.mean(0); Si = np.linalg.inv(np.cov(X.T))
D2c = np.array([mahalanobis(x,mu,Si)**2 for x in X])
D2m = MinCovDet(random_state=42).fit(X).mahalanobis(X)
thr = stats.chi2.ppf(0.975,2)

fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, D2, t in [(axes[0],D2c,'Classical'),(axes[1],D2m,'MCD')]:
    det = D2>thr
    ax.scatter(X[~det,0],X[~det,1],c='blue',s=20,alpha=0.5,label='정상')
    ax.scatter(X[det,0],X[det,1],c='red',s=40,marker='x',label='이상치')
    tp=((det)&(y==1)).sum(); fp=((det)&(y==0)).sum(); fn=((~det)&(y==1)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
    ax.set_title(f'{t}: P={p:.2f}, R={r:.2f}'); ax.legend()
plt.suptitle('마스킹 효과: 클러스터형 이상치'); plt.tight_layout(); plt.show()


**결과 해석**: Classical 방법은 이상치 클러스터가 평균/공분산을 끌어당겨 마스킹 발생. MCD는 로버스트 추정으로 이를 방지한다.

---
## 확장 토론

### 통계적 이상치 탐지 지침

1. 일변량: Grubbs → ESD
2. 다변량: MCD Mahalanobis 기본
3. 분포 가정 위반: 비모수 또는 ML 방법
4. 비율 > 50%: 붕괴점 초과, 문제 재정의 필요